# 1159. Market Analysis II

## Problem
We need to determine for each user whether the **brand of the second item they sold** matches their **favorite brand**.  
- If a user sold fewer than two items, the answer is **no**.  
- If the brand of the second sold item equals their favorite brand, the answer is **yes**.  
- Otherwise, the answer is **no**.  

---

## Schema

### Table: Users
| Column Name    | Type    | Description                          |
|----------------|---------|--------------------------------------|
| user_id        | INT     | Primary key, unique user identifier  |
| join_date      | DATE    | Date the user joined                 |
| favorite_brand | VARCHAR | User’s favorite brand                |

---

### Table: Orders
| Column Name   | Type | Description                                |
|---------------|------|--------------------------------------------|
| order_id      | INT  | Primary key, unique order identifier       |
| order_date    | DATE | Date the order was placed                  |
| item_id       | INT  | Foreign key referencing Items table        |
| buyer_id      | INT  | Foreign key referencing Users table (buyer)|
| seller_id     | INT  | Foreign key referencing Users table (seller)|

---

### Table: Items
| Column Name | Type    | Description                  |
|-------------|---------|------------------------------|
| item_id     | INT     | Primary key, unique item ID  |
| item_brand  | VARCHAR | Brand of the item            |

---

## Sample Data

### Users
| user_id | join_date  | favorite_brand |
|---------|------------|----------------|
| 1       | 2019-01-01 | Lenovo         |
| 2       | 2019-02-09 | Samsung        |
| 3       | 2019-01-19 | LG             |
| 4       | 2019-05-21 | HP             |

### Orders
| order_id | order_date | item_id | buyer_id | seller_id |
|----------|------------|---------|----------|-----------|
| 1        | 2019-08-01 | 4       | 1        | 2         |
| 2        | 2019-08-02 | 2       | 1        | 3         |
| 3        | 2019-08-03 | 3       | 2        | 3         |
| 4        | 2019-08-04 | 1       | 4        | 2         |
| 5        | 2019-08-04 | 1       | 3        | 4         |
| 6        | 2019-08-05 | 2       | 2        | 4         |

### Items
| item_id | item_brand |
|---------|------------|
| 1       | Samsung    |
| 2       | Lenovo     |
| 3       | LG         |
| 4       | HP         |

---

## Expected Output
| seller_id | 2nd_item_fav_brand |
|-----------|--------------------|
| 1         | no                 |
| 2         | yes                |
| 3         | yes                |
| 4         | no                 |

---

## PySpark Code: Create DataFrames and Temp Views

```python


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType
from datetime import date

# Schema for Users
users_schema = StructType([
    StructField("user_id", IntegerType(), False),
    StructField("join_date", DateType(), False),
    StructField("favorite_brand", StringType(), False)
])

# Schema for Orders
orders_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("order_date", DateType(), False),
    StructField("item_id", IntegerType(), False),
    StructField("buyer_id", IntegerType(), False),
    StructField("seller_id", IntegerType(), False)
])

# Schema for Items
items_schema = StructType([
    StructField("item_id", IntegerType(), False),
    StructField("item_brand", StringType(), False)
])

# Data for Users
users_data = [
    (1, date(2019,1,1), "Lenovo"),
    (2, date(2019,2,9), "Samsung"),
    (3, date(2019,1,19), "LG"),
    (4, date(2019,5,21), "HP")
]

# Data for Orders
orders_data = [
    (1, date(2019,8,1), 4, 1, 2),
    (2, date(2019,8,2), 2, 1, 3),
    (3, date(2019,8,3), 3, 2, 3),
    (4, date(2019,8,4), 1, 4, 2),
    (5, date(2019,8,4), 1, 3, 4),
    (6, date(2019,8,5), 2, 2, 4)
]

# Data for Items
items_data = [
    (1, "Samsung"),
    (2, "Lenovo"),
    (3, "LG"),
    (4, "HP")
]

# Create DataFrames
users_df = spark.createDataFrame(users_data, users_schema)
orders_df = spark.createDataFrame(orders_data, orders_schema)
items_df = spark.createDataFrame(items_data, items_schema)

# Register Temp Views
users_df.createOrReplaceTempView("Users")
orders_df.createOrReplaceTempView("Orders")
items_df.createOrReplaceTempView("Items")

# Quick check
users_df.show()
orders_df.show()
items_df.show()


In [0]:
%sql
with cte as (
  Select  u.user_id as seller_id  ,o.seller_id as o_seller_id  , o.order_date 
   , o.item_id  as sold_item_id , i.item_id as fav_item_id   
   ,dense_rank()over(partition by u.user_id order by o.order_date  asc ) rn 
   ,count( o.seller_id  )over(partition by u.user_id) as cnt
   from users  u left join Items i
  on u.favorite_brand  = i.item_brand 
  left join Orders o on 
  o.seller_id  = u.user_id 

)
,cte2 as (
Select seller_id  , 
case  when cnt < 2 then 'no'
when cnt >=2 and rn = 2 and sold_item_id <> fav_item_id then 'no'
when cnt >=2 and rn = 2 and sold_item_id = fav_item_id then 'yes'
else NULL end as 2nd_item_fav_brand 

 from cte
)
select * from cte2 where 2nd_item_fav_brand is not null order by seller_id  asc